# **SW01: Signals and Sampling**

<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: ADLS ISP |
<b>Version</b>: v1.2 <br><br>
<!-- 19.02.2025: Partially refactored. -->
</div>

In this notebook, we will discover how to represent and manipulate a discrete-time signal in Python. Discrete-time data is widespread in signal processing and is often used to represent time-varying data, such as audio or sensor signals. In the life sciences, other examples of time series data include the recording of EEG (electroencephalography) or ECG (electrocardiography) signals, or gene expression measured at regular time intervals, ... and soooooo many things more!

Here, we want to understand how such data is represented. Concepts we will encounter are: **Superposition**, **sampling** and **interpolation**.

<!--
## **Exercises**
* [Exercise 1 – Combining signals](#exercise1)  
* [Exercise 2 – Signal sampling and sampling rate](#exercise2)  
* [Exercise 3 – Signal representation with Pandas](#exercise3)  
* [Exercise 4a – Downsampling and interpolation](#exercise4a)  
* [Exercise 4b - Resampling and interpolation](#exercise4b)  
* [Exercise 5 - Sampling theorem and aliasing](#exercise5)  
-->

## **Preparations**

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import signal
from scipy.fft import fft, ifft, fftfreq, fftshift

# Jupyter / IPython configuration:
# Automatically reload modules when modified, if the extension is available
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]
%matplotlib inline

import sys
sys.path.insert(0, "../")
import isp
isp.setup_plotting()

Failed to read module file 'C:\Users\jmart\AppData\Local\Programs\Python\Python312\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\jmart\Documents\05_Studium\04_Programmierprojekte\04_Semester\ISP-Javier\venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jmart\Documents\05_Studium\04_Programmierprojekte\04_Semester\ISP-Javier\venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jmart\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_im

---

<a id='exercise1'></a>

## **&#9734; Exercise 1 – Combining signals**

We can use mathematical functions to represent a signal. A useful property of mathematical functions like $f(x)=x^2$ and $g(x)=\sin(x)$ is the ability to combine these functions using mathematical operations:

$$
\begin{align*}
f(x) + g(x) &= x^2 + \sin(x) \\ 
f(x) \cdot g(x) &= x^2\sin(x)
\end{align*}
$$

Consider a scenario where you have two audio signals representing different musical instruments. By adding these signals, we can create a new composite signal representing the combination of both instruments playing simultaneously. 

In signal processing, we frequently add signals together. The entire Fourier theory (which we will explore later) relies on this concept, and the [superposition principle](https://en.wikipedia.org/wiki/Superposition_principle) – applicable to many physical systems – is fundamentally based on it. More on these topics later.

In other scenarios, it may be necessary to multiply two signals. For instance, this technique is used in amplitude modulation (a method for transmitting messages via radio waves) and to create various special effects in music synthesizers or effects devices.


### **Instructions**
- Create two discrete-time signals, represented as a sequency samples $x_1[n]$ and $x_2[n]$
- Apply superposition to create a composite signal
- Visualize the original signals and the composite signal (using Matplotlib)


In [ ]:
######################
###    EXERCISE    ###
######################

# 1. Create signals
x1 = ...
x2 = ...

# 2. Create composite signal
x = ...

# 3. Visualize signals
...

In [ ]:
######################
###    Solution    ###
######################

# This is an exemplary solution. Of course you can choose different functions.

# Parameters
fs = 1000  # Sampling frequency
t = np.arange(0, 1, 1/fs)  # Time sequence
f1 = 1  # Frequency of first sine wave (Hz)
f2 = 20  # Frequency of second sine wave (Hz)

# Generate sine waves
x1 = np.sin(2 * np.pi * f1 * t)  # Sine wave 1
x2 = 0.2*np.sin(2 * np.pi * f2 * t)  # Sine wave 2

# Superimpose the signals (composite signal)
x = x1 + x2

# Plotting
plt.figure(figsize=(8, 4.5))
plt.plot(t, x1, label="Sine wave 1")
plt.plot(t, x2, label="Sine wave 2")
plt.plot(t, x, label="Superimposed signal", lw=3)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Signal superposition")
plt.legend()
plt.grid(True)
plt.show()

---

<a id='exercise2'></a>

## **&#9734; Exercise 2 – Signal sampling and sampling rate**

Signal sampling involves converting a continuous-time signal into a discrete-time signal. The sampling rate (also known as the sampling frequency) determines how many observations (samples) of the signal are taken per unit of time.

In the following, we assume that our continuous signal has the form $x(t) = A \sin(\omega t) = A\sin(2\pi f t)$, with frequency $f=5$ and amplitude $A=1$. We want to sample this signal with a sampling rate $f_s=100$ Hz.

In this exercise, we will illustrate the process of sampling. ...well, frankly, we are going to cheat a bit: To visualize the continuous-time signal, we will use a discrete-time representation, because that is what Matplotlib needs to plot the data. However, we will sample the continuous-time signal at a much higher rate so that it appears continuous in our visualization.


### **Instructions**
- Create samples that represent the "continuous-time" and discrete-time signals.  
Hint: Use [`np.linspace()`](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html) or [`np.arange()`](https://numpy.org/doc/stable/reference/generated/numpy.arange.html) in combination with `np.sin()`. Can you identify the step at which the sampling takes place?
- Visualize the continuous-time signal
- Also visualize the discrete-time signal. Hint: Use [`plt.stem()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.stem.html).

In [ ]:
######################
###    EXERCISE    ###
######################

# Parameters for the continuous signal
frequency = 5   # Frequency of the continuous signal (in Hz)
amplitude = 1   # Amplitude of the continuous signal
duration = 1    # Signal duration in seconds

# Parameters for our discrete signal representations
fs = 100        # Sampling rate (in Hz)
fs_cont = 1000  # Sampling rate for continuous signal (in Hz)

# 1. Create continuous signal
t_cont = ...
x_cont = ...

# 2. Create discrete signal
t_disc = ... 
x_disc = ...

# 3. Visualize signals
...

In [ ]:
######################
###    SOLUTION    ###
######################

# Parameters for the continuous signal
frequency = 5   # Frequency of the continuous signal (in Hz)
amplitude = 1   # Amplitude of the continuous signal
duration = 1    # Signal duration in seconds

# Parameters for our discrete signal representations
fs = 100        # Sampling rate (in Hz)
fs_cont = 1000  # Sampling rate for continuous signal (in Hz)

# 1) Samples representing the discretized signal (after sampling)
#################################################################
t_disc = np.arange(start=0, stop=duration, step=1/fs)
# Alternatively with np.linspace:
t_disc = np.linspace(start=0, stop=duration, num=duration*fs)
# Note: np.linspace and np.arange are almost equivalent. There is one small
#       difference: np.arange() excludes the stop value, while np.linspace() 
#       includes it. We could enforce the same behavior by using the endpoint
#       parameter of np.linspace(): np.linspace(..., endpoint=False)

# Next, we apply the sampling!
x_disc = amplitude * np.sin(2 * np.pi * frequency * t_disc)

# 2) Samples representing the continuous signal
###############################################
t_cont = np.arange(start=0, stop=duration, step=1/fs_cont) 
# Alternatively with np.linspace:
t_cont = np.linspace(start=0, stop=duration, num=duration*fs_cont)

# Next, we apply the sampling!
x_cont = amplitude * np.sin(2 * np.pi * frequency * t_cont)

# Print the number of samples
print("Number of samples:", len(x_disc))
print("Number of samples (continuous):", len(x_cont))

# 3) Visualize the signals
##########################
plt.figure(figsize=(7, 3))
plt.plot(t_cont, x_cont, label="Continuous Signal")
ret = plt.stem(t_disc, x_disc, "k", label="Discrete Samples")
plt.setp(ret[0], markersize=5, color="k", marker="o")   # Marker properties
plt.setp(ret[1], linewidth=0.7, color="k")              # Stem properties
plt.setp(ret[2], linewidth=0.7, color="k")              # Baseline properties

plt.setp(ret[0], markersize = 3)
plt.title("Signal Sampling")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1.02))
plt.grid(True)
plt.show()

# Fourier transform duality: The properties of the Fourier transform
# are mirrored in the time domain. 


---

<a id='exercise3'></a>
## **&#9734;&#9734; Exercise 3 – Signal representation with Pandas**

In this exercise, we want to have a look how we can represent time series data using Pandas. As you have certainly gotten to know previously, Pandas is a powerful library for data manipulation and analysis. It is particularly useful for time series data. 

We will create a simple time series data set and visualize it using Pandas. Concretely, we will model the [circadian rhythm](https://en.wikipedia.org/wiki/Circadian_rhythm) as a sinusoidal function. The illustration on the left shows an idealized circadian sequence – it is the signal we want to model here. The figure on the right displays a typical alertness level of a human, which is linked to the circadian rhythm. (Image sources: [Link1](http://dx.doi.org/10.1134/S2079057018040069), [Link2](https://sleepspace.com/circadian-rhythm-2/). An article on how to measure alertness: [Link](https://doi.org/10.1196/annals.1417.011))

<img src="../data/doc/circadian-rhythm-collage.jpg" alt="Circadian rhythm" style="max-height:300px;">

### **Tasks**

- Create a time sequence suitable to simulate a circadian rhythm using [`pd.date_range()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.date_range.html)
- Create [time series object](https://pandas.pydata.org/docs/user_guide/timeseries.html) using the time sequence as an index
- Use different sampling frequencies to display the sine signal

In [ ]:
######################
###    EXERCISE    ###
######################

# Time parameters
start_date = "2024-01-01"
start_date = "2024-01-01 00:00:00"
end_date = "2024-01-02 00:00:00"
freq = "min"

# 1. Create time sequence
time_index = pd.date_range(...)

# 2. Model the signal as sine wave
signal = ...

# 3. Create pd.Series or pd.DataFrame with the signal
df_circadian = ...

# 3. Visualize signal using pandas
df_circadian[...].plot()

In [ ]:
######################
###    SOLUTION    ###
######################

# Time parameters
start_date = "2024-01-01"
#start_date = "2024-01-01 00:00:00"
end_date = "2024-01-02 00:00:00"
freq = "min"    # Sampling frequency: 1 measurement per minute

# Create a DatetimeIndex between start and end date, with a frequency of 1 minute
time_index = pd.date_range(start=start_date, end=end_date, freq=freq)
# Alternative: Create a time_index usign date_range, for a start and a period of 1 day
#time_index = pd.date_range(start=start_date, periods=24*60, freq=freq)

# Generate circadian rhythm data (sinusoidal function)
frequency = 1  # Frequency of the circadian rhythm (how many cycles per day)
amplitude = 1  # Amplitude of the circadian rhythm

# Note: We cannot pass the DatetimeIndex object directly to the sine function,
#       as it is not immediately clear how to interpret the time values (do we
#       want to use seconds, minutes, hours, ...?). We need to first convert
#       the time values to a format that is suitable for the sine function.
t_days = (time_index.hour + time_index.minute/60)/24  # Time in days
circadian_rhythm_data = -amplitude * np.sin(2 * np.pi * frequency * t_days)

# Create a pd.DataFrame...
df_circadian = pd.DataFrame({"Time": time_index, "CR": circadian_rhythm_data})
df_circadian.set_index("Time", inplace=True)
# ...or a pd.Series
s_circadian = pd.Series(circadian_rhythm_data, index=time_index, name="CR")
s_circadian.index.name = "Time"

# Let's plot the circadian rhythm using pandas
s_circadian.plot(figsize=(8,4), linewidth=2, style="--", grid=True);

---

<a id='exercise4a'></a>
## **&#9734; Exercise 4a - Downsampling and interpolation**


Downsampling is a type of sampling, in which the sampling frequency of the available data is reduced. With interpolation, we can go the other way around: we can increase the sampling frequency of the available data. This is done by adding new data points between the existing ones. The new data points are calculated based on the existing ones. The process of adding new data points is called interpolation.

### **Instructions**
- Downsample the continuous signals (composite) from [Exercise 1](#exercise1). Use two methods
   - Method 1: Using numpy 
   - Method 2: Using pandas timeseries
- Visualize the original and downsampled signals
- Interpolate downsampled signals using various interpolation techniques. Make use of [`scipy.interpolate.interp1d()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html). Again use two methods
   - Method 1: Using the numpy array
   - Method 2: Using the pandas timeseries


In [ ]:
######################
###    EXERCISE    ###
######################

# Required for the interpolation
from scipy.interpolate import interp1d

# We use the data from exercise 1
print("x:", x.shape)
print("t:", t.shape)

# 1. Downsample the continuous signal
#    Use simply the slicing operator to select every 10th sample
downsampling_factor = 10
t_downsampled = ...
x_downsampled = ...

# 2. Interpolate the downsampled signal to reconstruct the original signal
interp_func = interp1d(...)
t_interp = np.linspace(t_downsampled.min(), t_downsampled.max(), len(t))
x_interp = ...

# 3. Plot the original, sampled and reconstructed signal
...

In [ ]:
######################
###    SOLUTION    ###
######################

# Required for the interpolation
from scipy.interpolate import interp1d

# We use the data from exercise 1
print("x:", x.shape)
print("t:", t.shape)

# Downsample the continuous signals
downsampling_factor = 10
x_downsampled = x[::downsampling_factor]
t_downsampled = t[::downsampling_factor]

# Interpolate downsampled signals using scipy
interp_linear = interp1d(t_downsampled, x_downsampled, kind="linear")
interp_cubic = interp1d(t_downsampled, x_downsampled, kind="cubic")
t_interpolation = np.linspace(t_downsampled.min(), t_downsampled.max(), len(t))

x_interp_lin = interp_linear(t_interpolation)
x_interp_cubic = interp_cubic(t_interpolation)

# Plotting
plt.figure(figsize=(8, 4))

# Original signals
plt.plot(t, x, linestyle="-", label="Original signal")

# Downsampled signals
ret = plt.stem(t_downsampled, x_downsampled, "k")
plt.setp(ret[0], markersize=5, color="k", marker="o")   # Marker properties
plt.setp(ret[1], linewidth=0.7, color="k")              # Stem properties
plt.setp(ret[2], linewidth=0.7, color="k")              # Baseline properties

# Interpolated signals
plt.plot(t_interpolation, x_interp_lin, label="Interpolated (linear)")
plt.plot(t_interpolation, x_interp_cubic, label="Interpolated (cubic)")

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Original, downsampled, and interpolated signals")
plt.legend()
plt.grid(True)
plt.xlim([0.4,0.8])
plt.show()

Note how well the original signal is already represented using cubic interpolation! Indeed, we can convince ourselves by looking at the error between original signal and the interpolated (reconstructed) signal.

In [ ]:
# Plot the error
x_resampled = interp1d(t, x, kind="linear")(t_interpolation)

error_lin = x_interp_lin - x_resampled
error_cubic = x_interp_cubic - x_resampled

print("Max. absolute error (full range):")
print("   Linear: %.3f" % np.max(np.abs(error_lin)))
print("   Cubic:  %.3f" % np.max(np.abs(error_cubic)))
# Clip the ends as the cubic interpolation has a larger error at the boundaries!
n_clip = int(5/100*len(t_interpolation))
print("Max. absolute error (clipped range):")
print("   Linear: %.3f" % np.max(np.abs(error_lin[n_clip:-n_clip])))
print("   Cubic:  %.3f" % np.max(np.abs(error_cubic[n_clip:-n_clip])))

plt.figure(figsize=(8, 4))
plt.plot(t_interpolation, error_lin, label="Error (linear)")
plt.plot(t_interpolation, error_cubic, label="Error (cubic)")
plt.xlabel("Time (s)")
plt.ylabel("Error")
plt.title("Interpolation error")
plt.legend()
plt.grid(True)
plt.xlim([0.4,0.8])
plt.show()

---

<a id='exercise4b'></a>
## **&#9734;&#9734; Exercise 4b – Resampling and interpolation**

We repeaet the exercise for our circadian rhythm data ([Exercise 3](#exercise3)). This time, however, we use the method [`pd.Series.resample()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.resample.html). Note that the resulting resampler object can be used to group multiple data points within a sampling interval. Here, we want to use [`Resampler.asfreq()`](https://pandas.pydata.org/docs/reference/api/pandas.core.resample.Resampler.asfreq.html) to reindex a Series/DataFrame with the given frequency without grouping. Alternatively, we could compute the [`Resampler.mean()`](https://pandas.pydata.org/docs/reference/api/pandas.core.resample.Resampler.mean.html) over the samples contained within a resampling period. 

For the interpolation, the resampler object also offers a method [`Resampler.interpolate()`](https://pandas.pydata.org/docs/reference/api/pandas.core.resample.Resampler.interpolate.html).

### **Instructions**
* Resample the circadian rhythm data using [`pd.Series.resample()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.resample.html)
* Reconstruct the original signal using [`pd.Resampler.interpolate()`](https://pandas.pydata.org/docs/reference/api/pandas.core.resample.Resampler.interpolate.html)
* Visualize the results

In [ ]:
######################
###    EXERCISE    ###
######################

# We use the data from exercise 3
print(df_circadian.head())

# 1. Resample the signal to every 3h hour
freq = "3h"
df_resampled_at_freq = df_circadian.resample(...).asfreq()
df_resampled_mean = df_circadian.resample(...).mean()

# 2. Apply interpolation
...

# 3. Visualize
...

In [ ]:
######################
###    SOLUTION    ###
######################

# Resample the signal to every 3h hour.
df_resampled_at_freq = df_circadian.resample("3h").asfreq()

# For comparison: Resample the signal to every 3h hour by taking the
# mean of the samples in each interval. (Check the comments below.)
df_resampled_mean = df_circadian.resample("3h").mean()
df_resampled_mean_causal = df_circadian.resample("3h", origin="end").mean()

# Plot the signals.
fig, ax = plt.subplots(figsize=(8, 4))
df_circadian.plot(linewidth=1, linestyle="-", ax=ax)
df_resampled_mean.plot(style = "o-", ax=ax)
# (Get the color of the last line)
color = ax.get_lines()[-1].get_c()
df_resampled_mean_causal.plot(style = "o:", color=color, ax=ax, 
                              markerfacecolor='none')
df_resampled_at_freq.plot(style = "o-", ax=ax)

# Decorate the plot.
ax.legend(["Circadian rhythm", 
           "Downsampled (mean)",
           "Downsampled (mean, causal)", 
           "Downsampled (asfreq)"])
xtick = pd.date_range(start=df_circadian.index.min(), 
                      end=df_circadian.index.max(), 
                      freq="1h")
ax.set_xticks(xtick, minor=True)
ax.grid("on", which="minor", axis="both", color="gray", alpha=0.05)
ax.grid("on", which="major", axis="both")

Note how the `mean()` signal runs ahead of the `asfreq()` signal. This is because the `mean()` method takes the mean of the samples in each resampling interval of 3 hours, while the `asfreq()` method simply takes the sample at the beginning of each interval. When taking the mean, we shift the signal to the left because we include information from the future. For instance at 3:00, we include information from 3:00 to 6:00. If we implement a mean filter for a real-time signal, we can only use data from the past (so-called causal filtering). In that situation, the signal would be shifted to the right. With pandas, we can enforce a causal filter by using the `origin` parameter of the `resample()` method.

All this demonstrates that processing (filtering) a time series may introduce a phase shift.

Next, we interpolate the sampled curves (`df_resampled_at_freq` and `df_resampled_mean`) such that we have samples every hour.

In [ ]:
# Interpolate the data
df_interp_asfreq= df_resampled_at_freq["CR"].resample("1h").interpolate(method="cubic")
df_interp_mean = df_resampled_mean["CR"].resample("1h").interpolate(method="cubic")

# Plot the signals.
fig, ax = plt.subplots(figsize=(8, 4))
df_circadian.plot(linewidth=1, linestyle="-", ax=ax)
df_interp_mean.plot(style = "o-", ax=ax)
df_interp_asfreq.plot(style = "d-", ax=ax)

# Decorate the plot.
ax.legend(["Circadian rhythm", "Downsampled (mean)", "Downsampled (asfreq)"])
xtick = pd.date_range(start=df_circadian.index.min(), 
                      end=df_circadian.index.max(), 
                      freq="1h")
ax.set_xticks(xtick, minor=True)
ax.grid("on", which="minor", axis="both", color="gray", alpha=0.05)
ax.grid("on", which="major", axis="both")

---

<a id='exercise5'></a>
## **&#9734; Exercis 5 – Sampling theorem and aliasing**


Aliasing is a phenomenon that occurs when a continuous signal is sampled at a rate lower (or equal) than twice its highest frequency component. This causes the high-frequency components to appear as lower frequencies in the sampled signal, creating distortion and ambiguity. To avoid aliasing, the sampling rate should be at least twice the Nyquist frequency, which is the highest frequency of interest in the signal.

## **Instructions**
* Create a sinusoid signal with frequency of $f = 1$ kHz
* Resample the signal with sampling frequency $f_s$ smaller than the Nyquist threshold
* Visualize the aliasing effect (you may have to adjust the visible range along the x-axis (`plt.xlim()`))

In [ ]:
######################
###    EXERCISE    ###
######################

# Define the sampling rate and the signal frequency
f_signal = 1000  # in Hz
f_sampling = ...  # in Hz 

# 1. Construct the discrete-time signal
f_data = 30000  # in Hz
t = ...
x = ...

# 2. Resample the signal
t_sampled = ...
x_sampled = ...

# 3. Visualize
...

In [ ]:
######################
###    SOLUTION    ###
######################

# Define the sampling rates and the signal frequency
f_data = 30000  # in Hz
f_signal = 1000  # in Hz
f_sampling = 1500  # in Hz

# Construct the discrete-time signal
t_signal = np.arange(0, 1, 1/f_data)
x_signal = np.sin(2 * np.pi * f_signal * t_signal)

# Resample the signal
t_undersampled = np.arange(0, 1, 1/f_sampling)
x_undersampled = np.sin(2 * np.pi * f_signal * t_undersampled)

# Plot the original and undersampled signals
plt.figure(figsize=(8, 3.5))
plt.plot(t_signal, x_signal)
plt.title("Aliasing")
plt.xlabel("Time (s)")
plt.ylabel("x(t)")

plt.plot(t_undersampled, x_undersampled)

ret = plt.stem(t_undersampled, x_undersampled, "k")
plt.setp(ret[0], markersize=5, color="k", marker="o")   # Marker properties
plt.setp(ret[1], linewidth=1, color="k")                # Stem properties
plt.setp(ret[2], linewidth=1, color="k")                # Baseline properties

plt.xlim([0,0.01])
plt.grid(True, axis="y")

plt.tight_layout()
plt.show()

Note how the sampled signal does not represent the original signal correctly. It appears to have a different frequency, which is caused by aliasing and the fact that the sampling frequency is too low to capture the original signal.